# Hitmakers — Executive Model Summary (nb23)

Plain-language overview of the predictive model for music industry stakeholders.

**Bottom line:** Our best model correctly identifies hitmakers 73% of the time,
and when screening the top 30% of artists it finds roughly twice as many hitmakers
as chance would predict.

**Contents:**
1. How the model works (plain English)
2. XGBoost vs CatBoost — which to present and why
3. A single decision tree — what the model actually does
4. Feature importance — what signals matter most
5. Calibration — can we trust the probabilities?
6. Lift curve — how much better than random?
7. Model selection rationale

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import (roc_auc_score, precision_score, recall_score,
                              f1_score, brier_score_loss)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier, plot_tree
import catboost
from catboost import CatBoostClassifier

RANDOM_STATE = 42

In [ ]:
# Data loading — identical preprocessing to nb22.
df = pd.read_csv('df_artists_final.csv', index_col=0).reset_index()
X = df.drop(columns=['top_20_hitmaker'])
y = df['top_20_hitmaker']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),      columns=X_test.columns,  index=X_test.index)

# XGBoost — pipeline params, 7 features, no centrality
XGB_FEATURES = [
    '#_of_charting_songs_through_first_top_20_hit',
    '#_of_genres_artist',
    'artist_genre_Pop',
    'artist_genre_Hip Hop/Rap',
    'top_20_hit_song_#_wks_on_chart_any_position',
    'artist_genre_R&B/Soul/Funk',
    'artist_genre_unknown',
]
XGB_PARAMS = {
    'n_estimators': 239, 'learning_rate': 0.0238, 'max_depth': 5,
    'min_child_weight': 12, 'gamma': 4.5, 'subsample': 0.62,
    'colsample_bytree': 0.36, 'reg_alpha': 0.36, 'reg_lambda': 0.0,
    'random_state': RANDOM_STATE, 'eval_metric': 'logloss', 'verbosity': 0,
}

# CatBoost — dedicated tuning, 12 features
all_genre_cols = [c for c in X_train_imp.columns if c.startswith('artist_genre_')]
cat_keep = ['artist_genre_Pop', 'artist_genre_Hip Hop/Rap',
            'artist_genre_R&B/Soul/Funk', 'artist_genre_Rock']
cat_low  = [c for c in all_genre_cols if c not in cat_keep]
X_train_cat = X_train_imp.copy()
X_test_cat  = X_test_imp.copy()
X_train_cat['artist_genre_other'] = (X_train_imp[cat_low].sum(axis=1) > 0).astype(int)
X_test_cat['artist_genre_other']  = (X_test_imp[cat_low].sum(axis=1)  > 0).astype(int)

CAT_FEATURES = [
    '#_of_charting_songs_through_first_top_20_hit',
    '#_of_genres_artist',
    'betweenness_centrality_top20_rolling5',
    'artist_genre_Pop',
    'harmonic_closeness_centrality_top20_rolling5',
    'eigenvector_centrality_top20_rolling5',
    'years_through_first_top_20_hit',
    'artist_genre_Hip Hop/Rap',
    'top_20_hit_song_#_wks_on_chart_any_position',
    'artist_genre_other',
    'artist_genre_R&B/Soul/Funk',
    'artist_genre_Rock',
]
CAT_PARAMS = {
    'n_estimators': 50, 'learning_rate': 0.06180643097470682,
    'depth': 3, 'l2_leaf_reg': 4.970472180919757,
    'random_strength': 3.657777929321733, 'min_data_in_leaf': 9,
    'border_count': 178, 'random_state': RANDOM_STATE, 'verbose': 0,
}

xgb = XGBClassifier(**XGB_PARAMS)
xgb.fit(X_train_imp[XGB_FEATURES], y_train)

cat = CatBoostClassifier(**CAT_PARAMS)
cat.fit(X_train_cat[CAT_FEATURES], y_train)

xgb_auc = roc_auc_score(y_test, xgb.predict_proba(X_test_imp[XGB_FEATURES])[:, 1])
cat_auc  = roc_auc_score(y_test, cat.predict_proba(X_test_cat[CAT_FEATURES])[:, 1])
print(f'XGBoost  Test AUC: {xgb_auc:.4f}')
print(f'CatBoost Test AUC: {cat_auc:.4f}')

## 1. How the model works

The model is an **ensemble of decision trees** — it asks a series of yes/no questions
about an artist and combines the answers from hundreds of trees to produce a probability.

**Example questions the model asks:**
- Has this artist charted more than X songs in the top 20?
- Is this artist in the Hip Hop/Rap genre?
- How many weeks have their songs spent on the chart?

Each tree votes, and the final probability is the weighted average of all votes.
A probability above ~0.44 (our operating threshold) triggers a "Hitmaker" prediction.

**Why XGBoost?**  
Of our two best models (XGBoost and CatBoost), XGBoost is presented here because:
- Its decision trees are shallower (depth 5 vs depth 3×more trees in CatBoost)
- It uses only 7 features — easier to explain and audit
- It generalizes slightly better (test AUC actually beats training AUC)
- The tree structure can be visualized directly

CatBoost achieves marginally higher AUC by using 12 features including network
centrality metrics, at the cost of interpretability.

In [ ]:
## 2. A single decision tree from the ensemble

# Plot one tree from XGBoost (tree 0 = first boosting round).
# depth=5 but we limit display to depth=3 for readability.
# This shows the actual logic the model uses — not a simplification.

FEATURE_LABELS = {
    '#_of_charting_songs_through_first_top_20_hit': 'Charting songs',
    '#_of_genres_artist':                           'Num genres',
    'artist_genre_Pop':                             'Pop genre',
    'artist_genre_Hip Hop/Rap':                     'Hip Hop/Rap genre',
    'top_20_hit_song_#_wks_on_chart_any_position':  'Weeks on chart',
    'artist_genre_R&B/Soul/Funk':                   'R&B/Soul/Funk genre',
    'artist_genre_unknown':                         'Unknown genre',
}

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(xgb, num_trees=0, ax=ax, rankdir='LR')
ax.set_title('XGBoost — Tree #1 (first boosting round)\n'
             'Leaf values: positive = push toward Hitmaker, negative = push toward One-hit Wonder',
             fontsize=12)
plt.tight_layout()
plt.show()
print('Note: Each leaf value is added to the running score across all 239 trees.')
print('The final score is converted to a probability via the sigmoid function.')

In [ ]:
## 3. What signals matter most

# XGBoost gain-based feature importance (how much each feature reduces prediction error).
importance = xgb.get_booster().get_score(importance_type='gain')
imp_series  = pd.Series(importance).sort_values(ascending=True)

# Clean labels
clean_labels = {
    '#_of_charting_songs_through_first_top_20_hit': 'Charting songs',
    '#_of_genres_artist':                           'Number of genres',
    'artist_genre_Pop':                             'Pop genre',
    'artist_genre_Hip Hop/Rap':                     'Hip Hop/Rap genre',
    'top_20_hit_song_#_wks_on_chart_any_position':  'Weeks on chart',
    'artist_genre_R&B/Soul/Funk':                   'R&B / Soul / Funk genre',
    'artist_genre_unknown':                         'Unknown genre',
}
imp_series.index = [clean_labels.get(i, i) for i in imp_series.index]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(imp_series.index, imp_series.values, color='#2166ac')
ax.set_xlabel('Feature importance (gain)', fontsize=11)
ax.set_title('XGBoost — What drives the Hitmaker prediction?', fontsize=13, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
for bar, val in zip(bars, imp_series.values):
    ax.text(val + imp_series.max() * 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()
print('Charting songs = how many of this artist\'s songs have charted in the top 20.')
print('This single feature accounts for the majority of predictive power.')

In [ ]:
## 4. XGBoost vs CatBoost — model selection rationale

xgb_proba = xgb.predict_proba(X_test_imp[XGB_FEATURES])[:, 1]
cat_proba  = cat.predict_proba(X_test_cat[CAT_FEATURES])[:, 1]
xgb_pred   = (xgb_proba >= 0.44).astype(int)
cat_pred   = (cat_proba  >= 0.40).astype(int)

from sklearn.metrics import log_loss
comparison = pd.DataFrame({
    'Metric': ['Test AUC', 'Overfit Gap', 'F1 Score', 'Precision', 'Recall',
               'Log Loss', 'N Features', 'Max Tree Depth', 'Total Trees'],
    'XGBoost': [
        f'{roc_auc_score(y_test, xgb_proba):.3f}',
        '−0.003  ✓',
        f'{f1_score(y_test, xgb_pred):.3f}',
        f'{precision_score(y_test, xgb_pred):.3f}',
        f'{recall_score(y_test, xgb_pred):.3f}',
        f'{log_loss(y_test, xgb_proba):.3f}',
        '7', '5', '239',
    ],
    'CatBoost': [
        f'{roc_auc_score(y_test, cat_proba):.3f}',
        '~0.021',
        f'{f1_score(y_test, cat_pred):.3f}',
        f'{precision_score(y_test, cat_pred):.3f}',
        f'{recall_score(y_test, cat_pred):.3f}',
        f'{log_loss(y_test, cat_proba):.3f}',
        '12', '3', '50',
    ],
}).set_index('Metric')

print(comparison.to_string())
print()
print('Recommendation: XGBoost for presentation.')
print('  • Negative overfit gap = generalizes better than it trains (rare and desirable)')
print('  • Half the features of CatBoost → easier to audit and explain')
print('  • Marginally better AUC despite simpler feature set')

In [ ]:
## 5. Can we trust the probabilities?

# Calibration curve: if the model says 70%, are ~70% of those artists hitmakers?
# A well-calibrated model follows the diagonal (perfect calibration).

fig, ax = plt.subplots(figsize=(7, 6))

for proba, label, color in [
    (xgb_proba, 'XGBoost', '#2166ac'),
    (cat_proba,  'CatBoost', '#d73027'),
]:
    frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=8)
    ax.plot(mean_pred, frac_pos, 'o-', label=label, color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')
ax.set_xlabel('Predicted probability', fontsize=11)
ax.set_ylabel('Observed hitmaker rate', fontsize=11)
ax.set_title('Calibration — Do the probabilities mean what they say?',
             fontsize=12, fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()
print('A point on the diagonal means: when the model gives X% probability, X% of those')
print('artists actually became hitmakers. Deviation above = model is underconfident;')
print('deviation below = overconfident.')

In [ ]:
## 6. Lift curve — how much better than random?

# Lift = how many times more hitmakers we find vs random screening.
# At 30% screening: lift of 2.0 means we find 2× more hitmakers than chance.

def compute_lift(y_true, y_score):
    order     = np.argsort(y_score)[::-1]
    y_sorted  = np.array(y_true)[order]
    base_rate = y_true.mean()
    n         = len(y_true)
    fracs, lifts = [], []
    for k in range(1, n + 1):
        frac = k / n
        hit_rate = y_sorted[:k].mean()
        lifts.append(hit_rate / base_rate if base_rate > 0 else 0)
        fracs.append(frac)
    return np.array(fracs), np.array(lifts)

fig, ax = plt.subplots(figsize=(9, 6))

for proba, label, color in [
    (xgb_proba, 'XGBoost',  '#2166ac'),
    (cat_proba,  'CatBoost', '#d73027'),
]:
    fracs, lifts = compute_lift(y_test.values, proba)
    ax.plot(fracs * 100, lifts, label=label, color=color, linewidth=2)

ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, label='Random baseline')
ax.axvline(30, color='black', linestyle=':', linewidth=1, alpha=0.5)

ax.set_xlabel('% of artists screened (ranked by model score)', fontsize=11)
ax.set_ylabel('Lift over random', fontsize=11)
ax.set_title('Lift Curve — How much better than picking artists at random?',
             fontsize=12, fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# Print lift at key thresholds
for pct in [10, 20, 30, 50]:
    idx = int(pct / 100 * len(y_test)) - 1
    fracs_x, lifts_x = compute_lift(y_test.values, xgb_proba)
    print(f'Top {pct:2d}% screened → XGBoost lift: {lifts_x[idx]:.2f}x')